<a href="https://colab.research.google.com/github/AarCho07/EDGE/blob/main/Test_4_Colab_Version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install gridstatus pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 551.2/551.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.4/159.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 

In [ ]:
"""
ISO-NE & NYISO Preliminary Real-Time Five-Minute LMP Downloader
----------------------------------------------------------------
Uses the open-source `gridstatus` library to pull ISO-NE and NYISO
5-minute LMP data automatically every 2 minutes. No account required.

Install dependencies:
    pip install gridstatus pandas

Usage:
    python isone_download_5min_lmp.py

Output:
    - Timestamped CSVs saved to ./downloads/
    - Master files: isone_5min_lmp_MASTER.csv, nyiso_5min_lmp_MASTER.csv

Location options (edit LOCATIONS below):
    "ALL"                  → every hub, zone, and interface
    ["H.INTERNAL_HUB"]    → ISO-NE Hub only
    [".Z.MAINE", ".Z.CONNECTICUT", ...]  → specific load zones

All available ISO-NE zones:
    .Z.MAINE, .Z.NEWHAMPSHIRE, .Z.VERMONT, .Z.CONNECTICUT,
    .Z.RHODEISLAND, .Z.SEMASS, .Z.WCMASS, .Z.NEMASSBOST
"""

import os
import sys
import time
import gridstatus
from datetime import datetime
from collections import deque
#When it's full and you add a new item, it automatically drops the oldest one.
#Used to track the last 5 downloaded file paths — so the list never grows beyond 5 no matter how long the script runs.

import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────

LOCATIONS       = "ALL"        # variable that tells gridstatus that we want to download data for everysingle pricing node, hub, and zone available.
OUTPUT_DIR      = "downloads_ISO"  # Defines the name of the folder where the files will go to
ISONE_MASTER    = os.path.join(OUTPUT_DIR, "isone_5min_lmp_MASTER.csv")
NYISO_MASTER    = os.path.join(OUTPUT_DIR, "nyiso_5min_lmp_MASTER.csv")
POLL_INTERVAL   = 120          # seconds between each fetch attempt (2 minutes)
RECENT_FILES_N  = 5            # how many recent downloaded files to display


# ── Master file helpers ───────────────────────────────────────────────────────

def append_to_master(df, master_path: str, time_col: str = "Time"): #

    #Append new rows to the master CSV. Dedupes on (Time + Location) so that running during the same 5-minute interval never adds duplicate rows. Returns: (min_time, max_time, total_row_count, was_duplicate)

    os.makedirs(OUTPUT_DIR, exist_ok=True) #Checks computer storage. If folder does not exist yet, it will create it. If it does exist, it ensures the code keeps running without crashing

    if os.path.exists(master_path): #looks to see if a masterfile has already been created from a prior run
        existing = pd.read_csv(master_path) #i
        # CRITICAL: parse Time immediately so it's a Timestamp, not a string.
        # Without this, drop_duplicates sees str != Timestamp and misses dupes.
        if time_col in existing.columns:
            existing[time_col] = pd.to_datetime(existing[time_col], utc=True)
        rows_before = len(existing) #counts and saves how many rows are in the master file before adding anything. Compares this to the count after adding, to detect whether new data was actually added or if it was all duplicates.
        combined = pd.concat([existing, df], ignore_index=True) # Glues the old historical data table and the brand-new downloaded data table together, creating one giant stacked table called combined
    else:
        rows_before = 0
        combined = df.copy()

    # Dedupe on Time + Location (same interval pulled twice = no new rows)
    dedupe_cols = [c for c in [time_col, "Location"] if c in combined.columns]
    if dedupe_cols:
        combined = combined.drop_duplicates(subset=dedupe_cols, keep="last")

    # Sort chronologically and convert timestamps one more time to be safe (in case anything slipped through
    if time_col in combined.columns:
        combined[time_col] = pd.to_datetime(combined[time_col], utc=True)
        combined = combined.sort_values(time_col)

    combined.to_csv(master_path, index=False)

    rows_after = len(combined)
    was_duplicate = (rows_after == rows_before)

    if time_col in combined.columns and len(combined) > 0:
        return combined[time_col].min(), combined[time_col].max(), rows_after, was_duplicate
    return None, None, rows_after, was_duplicate


def count_intervals(master_path: str, time_col: str = "Time") -> int:
    """Count how many distinct 5-minute snapshots are in a master file."""
    if not os.path.exists(master_path):
        return 0
    existing = pd.read_csv(master_path)
    if time_col not in existing.columns:
        return 0
    return existing[time_col].nunique()


# ── Downloaders ───────────────────────────────────────────────────────────────

def download_lmp():
    """Fetch latest ISO-NE RT 5-Min LMPs, save snapshot, append to master."""
    iso = gridstatus.ISONE()

    print("----------- NE ISO ----------")
    print("Fetching latest ISO-NE Preliminary Real-Time 5-Minute LMPs...")
    df = iso.get_lmp(date="latest", market="REAL_TIME_5_MIN", locations=LOCATIONS)

    if df is None or df.empty:
        raise ValueError("No data returned. The market may not be active right now / Check Connection.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filepath = os.path.join(OUTPUT_DIR, f"isone_5min_lmp_{timestamp}.csv")
    df.to_csv(filepath, index=False)

    isone_range = append_to_master(df, ISONE_MASTER)
    was_duplicate = isone_range[3]

    # If duplicate, delete the snapshot we just saved (it's identical to what's already in master, so keeping it would just waste disk space).
    # Safe to do because the master file is written and closed before this runs.
    if was_duplicate:
        os.remove(filepath)
        print(f"  (Duplicate interval — snapshot deleted, master unchanged)")
    else:
        print(f"✓ Saved {len(df)} rows → {filepath}")

    return filepath, df, isone_range, was_duplicate


def download_nyiso_lmp():
    """Fetch latest NYISO RT 5-Min LMPs, save snapshot, append to master."""
    nyiso = gridstatus.NYISO()

    print()
    print("----------- NY ISO ----------")
    print("Fetching latest NYISO Real-Time 5-Minute LMPs...")
    df = nyiso.get_lmp(date="latest", market="REAL_TIME_5_MIN", locations="ALL")

    if df is None or df.empty:
        raise ValueError("No NYISO data returned. The market may not be active right now.")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filepath = os.path.join(OUTPUT_DIR, f"nyiso_5min_lmp_{timestamp}.csv")
    df.to_csv(filepath, index=False)

    nyiso_range = append_to_master(df, NYISO_MASTER)
    was_duplicate = nyiso_range[3]

    if was_duplicate:
        os.remove(filepath)
        print(f"  (Duplicate interval — snapshot deleted, master unchanged)")
    else:
        print(f"✓ Saved {len(df)} rows → {filepath}")

    return filepath, df, nyiso_range, was_duplicate


# ── Previews ──────────────────────────────────────────────────────────────────

def preview(df, label: str, rows: int = 10, max_cols: int = None):
    """Unified preview for any ISO dataframe. Optionally cap columns."""
    display_df = df.iloc[:, :max_cols] if max_cols else df
    actual_rows = min(rows, len(df))
    col_count = len(display_df.columns)
    total_cols = len(df.columns)
    col_note = f", first {col_count} of {total_cols} columns" if max_cols else f", {total_cols} columns"
    print(f"\n── {label} Preview (first {actual_rows} of {len(df)} rows{col_note}) ──")
    print(display_df.head(rows).to_string(index=False))


# ── Screen control ───────────────────────────────────────────────────────────

def clear_screen():
    """
    Clear output in a way that works in both Google Colab and normal terminals.
    - Colab: uses IPython.display.clear_output (clears the cell output area)
    - Terminal (VS Code, PyCharm, standard shell): uses ANSI escape codes
    """
    try:
        from IPython.display import clear_output
        clear_output(wait=True)   # wait=True holds old output until new is ready
    except ImportError:
        # Not in IPython/Colab — use ANSI escape for regular terminals
        print("\033[2J\033[H", end="", flush=True)


# ── Summary printer ───────────────────────────────────────────────────────────

def print_summary(isone_range, nyiso_range,
                  isone_dup, nyiso_dup,
                  recent_isone, recent_nyiso,
                  last_run_time, run_count):

    fmt = "%B %d, %Y %I:%M:%S %p %Z"

    print()
    print("=" * 60)
    print("             MASTER DOCUMENTS SUMMARY")
    print(f"         (screen refreshed each cycle — run #{run_count})")
    print("=" * 60)

    # ── Recent files ──
    print("\n── Recent ISO-NE files downloaded ──")
    if recent_isone:
        for f in recent_isone:
            print(f"  {f}")
    else:
        print("  (none yet)")

    print("\n── Recent NYISO files downloaded ──")
    if recent_nyiso:
        for f in recent_nyiso:
            print(f"  {f}")
    else:
        print("  (none yet)")

    # ── Master doc ranges ──
    print()
    print("NE ISO MASTER DOC")
    if isone_range and isone_range[0] is not None:
        start, end, count, _ = isone_range
        print(f"  Spans:     {start.strftime(fmt)} → {end.strftime(fmt)}")
        print(f"  Rows:      {count} total rows")
        print(f"  Intervals: {count_intervals(ISONE_MASTER)} unique 5-minute snapshots")

    print()
    print("NY ISO MASTER DOC")
    if nyiso_range and nyiso_range[0] is not None:
        start, end, count, _ = nyiso_range
        print(f"  Spans:     {start.strftime(fmt)} → {end.strftime(fmt)}")
        print(f"  Rows:      {count} total rows")
        print(f"  Intervals: {count_intervals(NYISO_MASTER)} unique 5-minute snapshots")

    # ── Last run time ──
    print()
    print(f"Last execution: {last_run_time.strftime(fmt)}")

    # ── Duplicate / success notification (ALL CAPS, always last) ──
    print()
    if isone_dup and nyiso_dup:
        print("DUPLICATE DETECTED FOR BOTH ISO-NE AND NYISO — NO NEW DATA WAS ADDED TO EITHER MASTER FILE.")
    elif isone_dup:
        print("DUPLICATE DETECTED FOR ISO-NE — NO NEW DATA WAS ADDED TO THE ISO-NE MASTER FILE.")
        print("NYISO UPDATE SUCCESSFUL — NEW DATA HAS BEEN ADDED TO THE NYISO MASTER FILE.")
    elif nyiso_dup:
        print("DUPLICATE DETECTED FOR NYISO — NO NEW DATA WAS ADDED TO THE NYISO MASTER FILE.")
        print("ISO-NE UPDATE SUCCESSFUL — NEW DATA HAS BEEN ADDED TO THE ISO-NE MASTER FILE.")
    else:
        print("UPDATE SUCCESSFUL — NEW DATA HAS BEEN ADDED TO BOTH MASTER FILES.")

    print("=" * 60)


# ── Main loop ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    recent_isone = deque(maxlen=RECENT_FILES_N)   # keeps last N file paths
    recent_nyiso = deque(maxlen=RECENT_FILES_N)

    isone_range = nyiso_range = None
    isone_dup = nyiso_dup = False
    run_count = 0

    while True:
        run_count += 1
        last_run_time = datetime.now()

        # Clear screen at the start of every cycle so output replaces itself
        clear_screen()
        print(f"Scheduler running — cycle #{run_count} — fetching every {POLL_INTERVAL // 60} min.")
        print("Press Ctrl+C to stop.\n")

        # ── ISO-NE ──
        try:
            filepath, df, isone_range, isone_dup = download_lmp()
            preview(df, label="ISO-NE")
            if not isone_dup:
                recent_isone.append(filepath)
        except Exception as e:
            print(f"ISO-NE Error: {e}")
            isone_dup = False

        # ── NYISO ──
        try:
            nyiso_filepath, nyiso_df, nyiso_range, nyiso_dup = download_nyiso_lmp()
            preview(nyiso_df, label="NYISO", max_cols=10)
            if not nyiso_dup:
                recent_nyiso.append(nyiso_filepath)
        except Exception as e:
            print(f"NYISO Error: {e}")
            nyiso_dup = False

        # ── Summary (always last) ──
        print_summary(
            isone_range, nyiso_range,
            isone_dup, nyiso_dup,
            list(recent_isone), list(recent_nyiso),
            last_run_time, run_count,
        )

        print(f"\nSleeping {POLL_INTERVAL // 60} minutes until next fetch...\n")
        time.sleep(POLL_INTERVAL)

Scheduler running — cycle #1 — fetching every 2 min.
Press Ctrl+C to stop.

----------- NE ISO ----------
Fetching latest ISO-NE Preliminary Real-Time 5-Minute LMPs...
✓ Saved 1209 rows → downloads_ISO/isone_5min_lmp_20260703_181716.csv

── ISO-NE Preview (first 10 of 1209 rows, 10 columns) ──
                     Time            Interval Start              Interval End          Market            Location Location Type   LMP  Energy  Congestion  Loss
2026-07-03 14:15:00-04:00 2026-07-03 14:15:00-04:00 2026-07-03 14:20:00-04:00 REAL_TIME_5_MIN UN.FRNKLNSQ13.810CC  NETWORK NODE 50.47   49.17        1.86 -0.56
2026-07-03 14:15:00-04:00 2026-07-03 14:15:00-04:00 2026-07-03 14:20:00-04:00 REAL_TIME_5_MIN UN.FRNKLNSQ13.811CC  NETWORK NODE 50.47   49.17        1.86 -0.56
2026-07-03 14:15:00-04:00 2026-07-03 14:15:00-04:00 2026-07-03 14:20:00-04:00 REAL_TIME_5_MIN  UN.FRNKLNSQ13.89CC  NETWORK NODE 50.47   49.17        1.86 -0.56
2026-07-03 14:15:00-04:00 2026-07-03 14:15:00-04:00 2026-07-03 14

KeyboardInterrupt: 